# Hari 2 · Sesi 1 — Preprocessing: Bikin Data Siap Dimakan Model
Model machine learning cuma bisa makan **angka** dan **tidak boleh ada yang kosong**. Tugas preprocessing:
1. Pisahkan **target** (`y`) dan **fitur** (`X`), simpan ID test
2. **Feature engineering**: bikin kolom baru yang membantu
3. **Buang** kolom yang tidak berguna
4. **Isi** missing value
5. Ubah teks jadi angka (**encoding**)
6. Samakan kolom train & test, (opsional) **scaling**

> ⚠️ **Aturan emas:** semua "ukuran" (median, modus, rata-rata untuk scaling) dihitung dari **TRAIN**, lalu dipakai untuk train **dan** test. Jangan pernah menghitung dari test.

> ⚠️ **Jalankan tiap sel SEKALI saja, berurutan.** Sel yang mengubah data (drop, map, get_dummies) kalau dijalankan 2x bisa bikin error atau data rusak. Kalau kacau: menu **Run All** / **Restart & Run All**.

**Cara kerja tiap latihan**
1. Baca **Contoh**, lalu jalankan: klik selnya, tekan `Shift + Enter`.
2. Kerjakan **Latihan**: ganti `____` (atau `...`) dengan kode yang benar, lalu jalankan. Baris `cek(...)` di bawahnya langsung bilang ✅ atau ❌.
3. Mentok? Jalankan `hint('1.2')` di sel baru (bisa dipanggil berkali-kali, petunjuknya makin jelas). Masih mentok, baru `jawaban('1.2')`. Setelah lihat jawaban, **ketik ulang sendiri, jangan copy-paste**.
4. Kalau muncul error merah, baca **baris terakhirnya** dulu. Di bawahnya ada **💡 PENJELASAN DOJO** dalam bahasa Indonesia.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from cek import cek, hint, jawaban, progres

train = pd.read_csv('data/titanic/train.csv')
test = pd.read_csv('data/titanic/test.csv')
print('train:', train.shape, '| test:', test.shape)

train.head()

train: (712, 12) | test: (179, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q


## 1. Pisahkan target, fitur, dan ID

In [59]:
y = train['Survived']                    # target
X = train.drop(columns=['Survived'])     # fitur = semua kecuali target
X_test = test.copy()                     # test memang tidak punya kolom target
test_ids = test['PassengerId']           # simpan ID untuk file submission nanti
print(X.shape, y.shape, X_test.shape)

(712, 11) (712,) (179, 11)


## 2. Feature engineering (kolom baru)
Lakukan **sebelum** membuang kolom, karena kolom baru sering dibuat dari kolom yang nanti dibuang (misal `Cabin`).
Semua yang dilakukan ke `X` **wajib** dilakukan juga ke `X_test`.

In [60]:
X['FamilySize'] = X['SibSp'] + X['Parch'] + 1
X_test['FamilySize'] = X_test['SibSp'] + X_test['Parch'] + 1
X[['SibSp', 'Parch', 'FamilySize']].head(3)

,SibSp,Parch,FamilySize
0,1,0,2
1,1,0,2
2,0,0,1


**Latihan 3.1** — Buat kolom `IsAlone` (1 kalau `FamilySize == 1`, selain itu 0) di `X` **dan** `X_test`.

In [61]:
X['IsAlone'] = (X['FamilySize'] == 1).astype(int)
X_test['IsAlone'] = (X_test['FamilySize'] == 1).astype(int)
cek('3.1', X, X_test)
X.shape

✅ [3.1] Benar! Rapi!


(712, 13)

**Latihan 3.2** — `Cabin` kosong 77.5%, tapi **ada/tidaknya** kabin mungkin berguna. Buat kolom `HasCabin` (1 = ada isinya, 0 = kosong) di `X` dan `X_test`.

In [62]:
X['HasCabin'] = X['Cabin'].notnull().astype(int)
X_test['HasCabin'] = X_test['Cabin'].notnull().astype(int)
cek('3.2', X, X_test)
X.shape

✅ [3.2] Benar! Rapi!


(712, 14)

## 3. Buang kolom yang tidak berguna
Yang biasanya dibuang: **ID**, **teks bebas** yang hampir semua unik (nama, tiket), kolom yang **hampir kosong semua**, kolom yang isinya **cuma 1 nilai**.

In [63]:
# Contoh (tidak disimpan ke X, cuma memperlihatkan hasilnya):
X.drop(columns=['Ticket']).shape

(712, 13)

**Latihan 3.3** — Buang kolom `PassengerId`, `Name`, `Ticket`, `Cabin` dari `X` **dan** `X_test`. **Hasil drop harus disimpan** (`X = X.drop(...)`).

In [64]:
buang = ['PassengerId', 'Name', 'Ticket', 'Cabin']
X = X.drop(columns=buang)
X_test = X_test.drop(columns=buang)
cek('3.3', X, X_test)
X.shape

✅ [3.3] Benar! Aman!


(712, 10)

## 4. Isi missing value
**Angka** → median (tahan outlier). **Teks** → modus (nilai yang paling sering muncul).

⚠️ Di pandas 3 (yang terpasang di laptopmu), cara lama `X['Age'].fillna(median, inplace=True)` **diam-diam tidak mengubah apa-apa**. Selalu pakai pola **simpan kembali**:
```python
X['Age'] = X['Age'].fillna(median_age)
```

In [65]:
median_age = X['Age'].median()            # dihitung dari TRAIN
X['Age'] = X['Age'].fillna(median_age)
X_test['Age'] = X_test['Age'].fillna(median_age)   # test diisi pakai median TRAIN
print('Median umur train:', median_age)
print('Sisa kosong Age:', X['Age'].isnull().sum(), X_test['Age'].isnull().sum())

Median umur train: 28.5
Sisa kosong Age: 0 0


**Latihan 3.4** — `Fare` kosong 1 di **test**. Isi pakai **median Fare dari train** (untuk `X` dan `X_test`).

In [73]:
median_fare = X['Fare'].median()
X['Fare'] = X['Fare'].fillna(median_fare)
X_test['Fare'] = X_test['Fare'].fillna(median_fare)
cek('3.4', X, X_test)

✅ [3.4] Benar! Lanjut!


In [ ]:

# Contoh modus pada data mini
contoh = pd.Series(['S', 'C', 'S', None, 'Q', 'S'])
print(contoh.mode())       # mode() menghasilkan Series (bisa lebih dari satu modus)
print(contoh.mode()[0])    # ambil yang pertama -> 'S'

**Latihan 3.5** — Isi `Embarked` yang kosong dengan **modus dari train** (untuk `X` dan `X_test`).

In [ ]:
modus_embarked = X['Embarked'].mode()[____]
X['Embarked'] = X['Embarked'].fillna(modus_embarked)
X_test['Embarked'] = X_test['Embarked'].fillna(modus_embarked)
cek('3.5', X, X_test)

## 5. Encoding: ubah teks jadi angka
| Situasi | Cara |
|---|---|
| Cuma 2 nilai (laki/perempuan, ya/tidak) | `map` ke 0/1 |
| Beberapa nilai **tanpa urutan** (kota, pelabuhan) | **one-hot**: `pd.get_dummies` |
| Nilai **berurutan** (rendah < sedang < tinggi) | `map` ke 0, 1, 2 |

In [ ]:
# Contoh map pada data mini
ukuran = pd.Series(['kecil', 'besar', 'sedang', 'kecil'])
ukuran.map({'kecil': 0, 'sedang': 1, 'besar': 2})

**Latihan 3.6** — Ubah `Sex` jadi angka: `'male'` → 0, `'female'` → 1 (untuk `X` dan `X_test`).

In [74]:
X['Sex'] = X['Sex'].map({'male':0 , 'female': 1})
X_test['Sex'] = X_test['Sex'].map({'male': 0, 'female': 1})
cek('3.6', X, X_test)

✅ [3.6] Benar! Lanjut!


In [ ]:
# Contoh one-hot pada data mini: tiap kategori jadi kolom 0/1 sendiri
kota = pd.DataFrame({'kota': ['Jakarta', 'Bandung', 'Jakarta']})
pd.get_dummies(kota, columns=['kota'], dtype=int)

**Latihan 3.7** — One-hot kolom `Embarked` di `X` dan `X_test` (pakai `dtype=int` supaya hasilnya 0/1).

In [ ]:
X = pd.get_dummies(X, columns=[____], dtype=int)
X_test = pd.get_dummies(X_test, columns=[____], dtype=int)
cek('3.7', X, X_test)

### Samakan kolom train & test
Bahaya one-hot: kalau test **tidak punya** suatu kategori (atau punya kategori baru), kolom hasilnya bisa beda dengan train, lalu model error. Solusinya, paksa `X_test` punya kolom yang **persis sama** dengan `X`:

**Latihan 3.8** — Samakan kolom `X_test` dengan `X`. Kolom yang tidak ada di test diisi **0**.

In [ ]:
X_test = X_test.reindex(columns=X.columns, fill_value=____)
cek('3.8', X, X_test)

## 6. Scaling (opsional)
Menyamakan skala angka (rata-rata 0, standar deviasi 1). **Perlu** untuk model berbasis jarak/garis: Logistic/Linear Regression, KNN, SVM. **Tidak perlu** untuk model pohon: Decision Tree, Random Forest, Gradient Boosting.

Aturan emas berlaku lagi: `fit_transform` di train (belajar rata-rata & std), `transform` saja di test.

**Latihan 3.9** — Standarkan kolom `Age`, `Fare`, `FamilySize`.

In [ ]:
from sklearn.preprocessing import StandardScaler
kolom_skala = ['Age', 'Fare', 'FamilySize']
scaler = StandardScaler()
X[kolom_skala] = scaler.____(X[kolom_skala])
X_test[kolom_skala] = scaler.____(X_test[kolom_skala])
cek('3.9', X, X_test)

## 7. Pemeriksaan akhir (wajib sebelum modeling!)
Tiga syarat data siap: **tidak ada yang kosong**, **semua kolom angka**, **kolom train = kolom test**.

**Latihan 3.10** — Tulis sendiri tiga pemeriksaannya (jangan scroll ke atas).

In [ ]:
sisa_kosong = ...    # total missing di X + X_test (harusnya 0)
kolom_teks = ...     # daftar kolom non-angka di X (harusnya [])
kolom_sama = ...     # True kalau daftar & urutan kolom X dan X_test sama persis
cek('3.10', sisa_kosong, kolom_teks, kolom_sama)

In [ ]:
X.head()

## Bonus (boleh dilewati): trik yang sering menaikkan skor

In [ ]:
# 1) Gelar dari nama (Mr, Mrs, Miss, Master) -> sering jadi fitur kuat di Titanic
contoh = train[['Name']].copy()
contoh['Title'] = contoh['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
contoh['Title'].value_counts().head(8)

In [ ]:
# 2) Data miring kanan (Fare, harga, pendapatan) -> log supaya lebih "normal"
print('Sebelum:', train['Fare'].skew().round(2), '| Sesudah log1p:', np.log1p(train['Fare']).skew().round(2))

In [ ]:
# 3) Nilai mustahil -> jadikan kosong (NaN), nanti diisi seperti missing biasa
contoh_umur = pd.Series([25, 0, 41, 150, 33])
contoh_umur = contoh_umur.where((contoh_umur > 0) & (contoh_umur < 100))   # selain itu jadi NaN
contoh_umur

In [ ]:
# 4) Ejaan kategori tidak konsisten: 'jakarta', 'JAKARTA', 'Jakarta ' -> 'Jakarta'
kota = pd.Series(['Jakarta', 'jakarta', 'JAKARTA', 'Jakarta ', 'Bandung'])
kota.str.strip().str.title().value_counts()

In [ ]:
progres()